In [ ]:
# trying out a new evaluation fucntion for social
import sys
import os
sys.path.append(os.path.abspath('../..'))
from utlis.vis_valid_utlis.scom_traga_utlis import com_distance_qc

label1 = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30"
# '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_04_10/20241212V1RE1L23Fmini_p20241212V1BE1L23F'

# label1 = '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_01/20250324PMCBmini_psamecageLE2_15_09'

# label1 = '/data/big_rim/rsync_dcc_sum/Oct3V1/2025_03_11/20241224PMCLE1mini_p20240303PMC2_14_44'
out = com_distance_qc(label1, com_folder_name='yolo_tracking/predict_train_lone_social_retrain_0917', jump_k=400, show_plot=True) #prefer_filtered=True,  jump_k=6,  first_n=3000,

out['jump_frames']

In [ ]:


import os
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from scipy.optimize import least_squares


def plot_com_circle_for_path_social(
    base_path,
    pred_folder='DANNCE/predict00',
    com_dir='yolo_tracking/predict_train_lone_social_retrain_0917',
    vis_dir='COM/predict00/vis',
    filename=None
):
    """
    Load multi-animal COM (x,y,z), extract x/y for each animal,
    fit circles to each convex hull in x-y space,
    plot both COM trajectories in distinct colors,
    and save under base_path/vis/.

    Fallbacks:
      1) base_path/pred_folder/com3d_used.mat
      2) base_path/COM/predict00/com3d0.mat
    """
    # determine COM file
    primary = os.path.join(base_path, pred_folder, 'com3d_used.mat')
    fallback = os.path.join(base_path, com_dir, 'com3d0.mat')
    if os.path.isfile(primary):
        com_file = primary
    elif os.path.isfile(fallback):
        com_file = fallback
        print(f"Primary COM file not found; using fallback: {com_file}")
    else:
        raise FileNotFoundError(f"Neither {primary} nor {fallback} exists.")

    # load raw COM
    mat = sio.loadmat(com_file)
    raw = mat.get('com')
    if raw is None:
        raise KeyError(f"'com' variable not found in {com_file}")

    # reshape to (frames,2,n_animals)
    if raw.ndim == 3 and raw.shape[1] == 3:
        # already (frames,3,n); take x,y only
        com_xy = raw[:, :2, :]
    elif raw.ndim == 3 and raw.shape[1] == 2:
        # already (frames,2,n)
        com_xy = raw
    elif raw.ndim == 2 and raw.shape[1] % 3 == 0:
        # flattened (frames,3*n)
        n = raw.shape[1] // 3
        com3 = raw.reshape(-1, 3, n)
        com_xy = com3[:, :2, :]
    elif raw.ndim == 2 and raw.shape[1] % 2 == 0:
        # flattened (frames,2*n)
        n = raw.shape[1] // 2
        com_xy = raw.reshape(-1, 2, n)
    else:
        raise ValueError(
            f"Unexpected COM array shape {raw.shape},"
            " expected (frames,3,n) or (frames,3*n) or (frames,2,n) or (frames,2*n)"
        )

    frames, _, n_animals = com_xy.shape
    if n_animals < 2:
        raise ValueError(f"Expected at least 2 animals, found {n_animals}")

    # prepare plot
    fig, ax = plt.subplots(figsize=(8, 8))

    # color list for animals
    colors = ['C0', 'C1']

    def residuals(params, xh, yh):
        return np.hypot(xh - params[0], yh - params[1]) - params[2]

    # plot each animal's COM and circle fit
    for i in range(min(n_animals, 2)):
        x = com_xy[:, 0, i]
        y = com_xy[:, 1, i]
        # plot trajectory
        ax.scatter(x, y, color=colors[i], s=10, alpha=0.6, label=f'COM{i+1}')
        # convex hull and circle
        pts = np.stack([x, y], axis=1)
        hull = ConvexHull(pts)
        hx, hy = pts[hull.vertices, 0], pts[hull.vertices, 1]
        init = [hx.mean(), hy.mean(), np.mean(np.hypot(hx - hx.mean(), hy - hy.mean()))]
        res = least_squares(residuals, init, args=(hx, hy))
        xc, yc, r = res.x
        theta = np.linspace(0, 2*np.pi, 200)
        # ax.plot(xc + r*np.cos(theta), yc + r*np.sin(theta),
        #         linestyle='--', lw=2, color=colors[i],
        #         label=f'Fit{i+1}')
        # ax.scatter(xc, yc, s=50, marker='x', color=colors[i],
        #            label=f'Center{i+1}')

    ax.set_aspect('equal', 'box')
    ax.set_title(os.path.basename(base_path))
    ax.legend(loc='best', fontsize='small')

    # # save
    # save_folder = os.path.join(base_path, vis_dir)
    # os.makedirs(save_folder, exist_ok=True)
    # if filename is None:
    #     filename = 'com_circle.png'
    # save_path = os.path.join(save_folder, filename)
    # plt.tight_layout()
    # plt.savefig(save_path, dpi=300)
    # plt.close(fig)
    # print(f"Saved COM circle plot to: {save_path}")


In [ ]:
p = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30"
plot_com_circle_for_path_social(p)

In [ ]:
import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import matplotlib.cm as cm

def plot_com_social_with_time_and_meetings(
    base_path,
    pred_folder='DANNCE/predict00',
    com_dir='yolo_tracking/predict_train_lone_social_retrain_0917',
    vis_dir='COM/predict00/vis',
    filename_xy='com_xy_events.png',
    filename_t='distance_time.png',
    events_csv='meeting_events.csv',
    pair=(0, 1),
    fps=30.0,
    time=None,
    dist_thresh=120.0,
    min_event_frames=15,
    merge_gap_frames=10,
    # XY style (event-focused)
    animal_colors=('C0','C1'),
    base_alpha=0.18,           # faint full-path alpha
    base_lw=1.0,               # faint full-path linewidth
    highlight_window_s=0.75,   # expand meeting window by ± seconds
    highlight_lw=2.2,          # linewidth for highlighted segments
    event_cmap='tab10',
    annotate_meetings=True,
    show=True,
    save=False,
):
    """
    Computes inter-animal XY distance and 'meeting' events (distance <= dist_thresh).
    XY panel: faint full paths per animal (fixed color), with thick colored segments
    around each meeting window (±highlight_window_s). Numbers tie to the distance plot.
    Returns dict(time, dist_xy, events, mask).
    """

    # ---------- 1) locate COM ----------
    primary = os.path.join(base_path, pred_folder, 'com3d_used.mat')
    fallback = os.path.join(base_path, com_dir, 'com3d0.mat')
    com_file = primary if os.path.isfile(primary) else fallback
    if not os.path.isfile(com_file):
        raise FileNotFoundError(f"Neither {primary} nor {fallback} exists.")
    if com_file == fallback:
        print(f"Primary COM file not found; using fallback: {com_file}")

    # ---------- 2) load to (T,2,N) ----------
    raw = sio.loadmat(com_file).get('com')
    if raw is None:
        raise KeyError(f"'com' variable not found in {com_file}")

    if raw.ndim == 3 and raw.shape[1] == 3:
        com_xy = raw[:, :2, :]
    elif raw.ndim == 3 and raw.shape[1] == 2:
        com_xy = raw
    elif raw.ndim == 2 and raw.shape[1] % 3 == 0:
        N = raw.shape[1] // 3
        com_xy = raw.reshape(-1, 3, N)[:, :2, :]
    elif raw.ndim == 2 and raw.shape[1] % 2 == 0:
        N = raw.shape[1] // 2
        com_xy = raw.reshape(-1, 2, N)
    else:
        raise ValueError(f"Unexpected COM shape {raw.shape}")

    T, _, N = com_xy.shape
    if max(pair) >= N:
        raise ValueError(f"Pair index {pair} exceeds #animals {N}")

    # ---------- 3) time ----------
    if time is None:
        t = np.arange(T) / float(fps)
    else:
        t = np.asarray(time)
        if t.shape[0] != T:
            raise ValueError(f"time length {t.shape[0]} != frames {T}")

    # ---------- 4) coords + distance ----------
    x1, y1 = com_xy[:, 0, pair[0]], com_xy[:, 1, pair[0]]
    x2, y2 = com_xy[:, 0, pair[1]], com_xy[:, 1, pair[1]]
    valid = np.isfinite(x1) & np.isfinite(y1) & np.isfinite(x2) & np.isfinite(y2)

    dist_xy = np.full(T, np.nan)
    dist_xy[valid] = np.hypot(x1[valid] - x2[valid], y1[valid] - y2[valid])

    base_mask = (dist_xy <= dist_thresh)
    base_mask[~valid] = False

    # ---------- 5) segments (robust) ----------
    def _segments_from_mask(mask: np.ndarray) -> np.ndarray:
        m = np.asarray(mask, dtype=bool)
        if m.size == 0:
            return np.empty((0, 2), dtype=int)
        starts = np.flatnonzero(~np.concatenate(([False], m[:-1])) & m)
        ends   = np.flatnonzero(m & ~np.concatenate((m[1:], [False])))
        return np.column_stack((starts, ends)) if starts.size else np.empty((0, 2), int)

    def _filter_and_merge(segs: np.ndarray, min_len: int, gap: int) -> np.ndarray:
        if segs.size == 0:
            return segs
        lengths = segs[:, 1] - segs[:, 0] + 1
        segs = segs[lengths >= min_len]
        if segs.size == 0:
            return segs
        merged = [segs[0].tolist()]
        for s, e in segs[1:]:
            if s - merged[-1][1] - 1 <= gap:
                merged[-1][1] = e
            else:
                merged.append([s, e])
        return np.asarray(merged, dtype=int)

    segs = _filter_and_merge(_segments_from_mask(base_mask), min_event_frames, merge_gap_frames)

    # ---------- 6) event table ----------
    events = []
    for s, e in segs:
        w = slice(s, e + 1)
        finite = np.isfinite(dist_xy[w])
        idx_rep = (s + e) // 2
        if finite.any():
            idx_rep = s + int(np.nanargmin(dist_xy[w]))
        events.append(dict(
            start_idx=int(s),
            end_idx=int(e),
            rep_idx=int(idx_rep),
            start_time=float(t[s]),
            end_time=float(t[e]),
            rep_time=float(t[idx_rep]),
            duration_s=float(t[e] - t[s]),
            min_distance=float(dist_xy[idx_rep]) if np.isfinite(dist_xy[idx_rep]) else np.nan,
        ))

    # ---------- helpers for XY line drawing ----------
    def _line_segments(x, y, mask_pairs):
        # segments between (i,i+1) where both frames valid
        pts = np.column_stack([x, y])
        segs = np.stack([pts[:-1], pts[1:]], axis=1)
        return segs[mask_pairs]

    valid_seg = valid[:-1] & valid[1:]
    cmap_ev = cm.get_cmap(event_cmap, max(1, len(events)))
    pad = int(round(highlight_window_s * fps))

    # ---------- 7A) XY: faint full path + colored meeting highlights ----------
    fig_xy, ax_xy = plt.subplots(figsize=(7, 7))

    # faint base (full trajectory)
    if valid_seg.any():
        segs1_all = _line_segments(x1, y1, valid_seg)
        segs2_all = _line_segments(x2, y2, valid_seg)
        ax_xy.add_collection(LineCollection(segs1_all, colors=animal_colors[0], linewidths=base_lw, alpha=base_alpha))
        ax_xy.add_collection(LineCollection(segs2_all, colors=animal_colors[1], linewidths=base_lw, alpha=base_alpha))

    # highlight windows around each meeting
    for k, ev in enumerate(events, start=1):
        col = cmap_ev(k-1)
        s = max(0, ev['start_idx'] - pad)
        e = min(T-1, ev['end_idx'] + pad)
        win_seg = np.zeros_like(valid_seg)
        win_seg[s:e] = True
        mask1 = valid_seg & win_seg
        if mask1.any():
            ax_xy.add_collection(LineCollection(_line_segments(x1, y1, mask1),
                                                colors=col, linewidths=highlight_lw, alpha=0.95))
            ax_xy.add_collection(LineCollection(_line_segments(x2, y2, mask1),
                                                colors=col, linewidths=highlight_lw, alpha=0.95))
        
        # numbered midpoint (respect annotate_meetings)
        if annotate_meetings:
            idx = ev['rep_idx']
            if 0 <= idx < T and valid[idx]:
                mx, my = (x1[idx] + x2[idx]) * 0.5, (y1[idx] + y2[idx]) * 0.5
                ax_xy.text(mx, my, str(k), color='k', fontsize=9, ha='center', va='center',
                        bbox=dict(boxstyle="circle,pad=0.15", fc='white', ec=col, lw=0.9, alpha=0.95))
        
        # numbered midpoint
        # idx = ev['rep_idx']
        # if 0 <= idx < T and valid[idx]:
        #     mx, my = (x1[idx] + x2[idx]) * 0.5, (y1[idx] + y2[idx]) * 0.5
        #     ax_xy.text(mx, my, str(k), color='k', fontsize=9, ha='center', va='center',
        #                bbox=dict(boxstyle="circle,pad=0.15", fc='white', ec=col, lw=0.9, alpha=0.95))

    # mark first valid frame per animal
    if valid.any():
        f0 = np.flatnonzero(valid)[0]
        ax_xy.plot([x1[f0]], [y1[f0]], marker='o', ms=4, mfc='C0', mec='none')
        ax_xy.plot([x2[f0]], [y2[f0]], marker='o', ms=4, mfc='C1', mec='none')
        # bewlow i slike start with black circle??
        # ax_xy.plot([x1[f0]], [y1[f0]], marker='o', ms=6, mfc='none', mec='k')
        # ax_xy.plot([x2[f0]], [y2[f0]], marker='o', ms=6, mfc='none', mec='k')

    ax_xy.set_aspect('equal', 'box')
    ax_xy.set_title(os.path.basename(base_path))
    ax_xy.set_xlabel('x'); ax_xy.set_ylabel('y')
    # legend proxies
    proxy1 = Line2D([0],[0], color=animal_colors[0], lw=2)
    proxy2 = Line2D([0],[0], color=animal_colors[1], lw=2)
    ax_xy.legend([proxy1, proxy2], [f'COM{pair[0]+1}', f'COM{pair[1]+1}'], fontsize='small', loc='best')

    # ---------- 7B) distance–time with colored spans ----------
    fig_t, ax_t = plt.subplots(figsize=(10, 3))
    ax_t.plot(t, dist_xy, lw=1.2, color='0.15')
    ax_t.axhline(dist_thresh, ls='--', lw=1, color='k', alpha=0.7)

    y_max = np.nanmax(dist_xy) if np.any(np.isfinite(dist_xy)) else dist_thresh
    y_text = 0.9 * y_max if np.isfinite(y_max) else dist_thresh
    for k, ev in enumerate(events, start=1):
        s, e = ev['start_idx'], ev['end_idx']
        col = cmap_ev(k-1)
        ax_t.axvspan(t[s], t[e], color=col, alpha=0.18)
        if annotate_meetings:
            ax_t.text(ev['rep_time'], y_text, str(k), color='k', fontsize=9,
                      ha='center', va='center',
                      bbox=dict(boxstyle="circle,pad=0.15", fc='white', ec=col, lw=0.9, alpha=0.9))

    ax_t.set_xlabel('time (s)')
    ax_t.set_ylabel('distance (xy)')
    ax_t.set_title('Inter-animal distance (xy)')

    # ---------- 8) I/O ----------
    if save:
        out_dir = os.path.join(base_path, vis_dir)
        os.makedirs(out_dir, exist_ok=True)
        fig_xy.tight_layout(); fig_t.tight_layout()
        fig_xy.savefig(os.path.join(out_dir, filename_xy), dpi=300)
        fig_t.savefig(os.path.join(out_dir, filename_t), dpi=300)
        if events_csv:
            import csv
            with open(os.path.join(out_dir, events_csv), 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=[
                    'start_idx','end_idx','rep_idx','start_time','end_time','rep_time','duration_s','min_distance'
                ])
                w.writeheader(); w.writerows(events)

    if not show:
        plt.close(fig_xy); plt.close(fig_t)

    return dict(time=t, dist_xy=dist_xy, events=events, mask=base_mask)


In [ ]:
out = plot_com_social_with_time_and_meetings(
    base_path=p,
    dist_thresh=100,#100.0,   # adjust manually
    # time_alpha_min = 0.01,
    # time_alpha_max = 0.5,
    annotate_meetings=False,
    base_alpha = 0.65,
    save=False,          # nothing savedexampl
    show=True            # still pops up the figures
)


In [ ]:
import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.cm as cm

def _load_com_xy(base_path,
                 pred_folder='DANNCE/predict00',
                 com_dir='yolo_tracking/predict_train_lone_social_retrain_0917'):
    p1 = os.path.join(base_path, pred_folder, 'com3d_used.mat')
    p2 = os.path.join(base_path, com_dir, 'com3d0.mat')
    com_file = p1 if os.path.isfile(p1) else p2
    raw = sio.loadmat(com_file).get('com')
    if raw is None: raise KeyError(f"'com' not found in {com_file}")
    if raw.ndim == 3 and raw.shape[1] == 3:  # (T,3,N)
        return raw[:, :2, :]
    if raw.ndim == 3 and raw.shape[1] == 2:  # (T,2,N)
        return raw
    if raw.ndim == 2 and raw.shape[1] % 3 == 0:
        N = raw.shape[1] // 3
        return raw.reshape(-1, 3, N)[:, :2, :]
    if raw.ndim == 2 and raw.shape[1] % 2 == 0:
        N = raw.shape[1] // 2
        return raw.reshape(-1, 2, N)
    raise ValueError(f"Unexpected COM shape {raw.shape}")

def _events_from_dist(x1, y1, x2, y2, fps, dist_thresh=120.0,
                      min_len=15, gap=10, pad_s=0.75):
    T = len(x1)
    valid = np.isfinite(x1)&np.isfinite(y1)&np.isfinite(x2)&np.isfinite(y2)
    t = np.arange(T)/float(fps)
    dist = np.full(T, np.nan)
    dist[valid] = np.hypot(x1[valid]-x2[valid], y1[valid]-y2[valid])
    close = np.isfinite(dist) & (dist <= dist_thresh)

    starts = np.flatnonzero(~np.concatenate(([False], close[:-1])) & close)
    ends   = np.flatnonzero(close & ~np.concatenate((close[1:], [False])))
    segs = np.column_stack([starts, ends]) if starts.size else np.empty((0,2), int)
    if segs.size:
        L = segs[:,1]-segs[:,0]+1
        segs = segs[L>=min_len]
        merged=[]
        for s,e in segs:
            if merged and s-merged[-1][1]-1 <= gap: merged[-1][1]=e
            else: merged.append([s,e])
        segs = np.asarray(merged, int) if merged else np.empty((0,2), int)

    events=[]
    for s,e in segs:
        w = slice(s, e+1)
        rep = s + int(np.nanargmin(dist[w])) if np.any(np.isfinite(dist[w])) else (s+e)//2
        events.append(dict(start=s, end=e, rep=rep, t0=t[s], t1=t[e], trep=t[rep]))
    pad = int(round(pad_s*fps))
    return t, dist, valid, events, pad

def _seg_lines(x, y, mask_pairs):
    pts = np.column_stack([x, y])
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    return seg[mask_pairs]

def plot_social_quad_poster(
    base_path,
    pair=(0,1), fps=30.0, dist_thresh=120.0,
    px_to_mm=None,
    animal_colors=('C0','C1'),
    annotate_meetings=False,         # still honored
    show=True, save=False, out_png='quad_poster.png'
):
    com = _load_com_xy(base_path)
    T, _, N = com.shape
    if max(pair) >= N: raise ValueError("pair index out of range")

    x1,y1 = com[:,0,pair[0]], com[:,1,pair[0]]
    x2,y2 = com[:,0,pair[1]], com[:,1,pair[1]]

    if px_to_mm is not None:
        s=float(px_to_mm); x1*=s; y1*=s; x2*=s; y2*=s
        xlab, ylab_xy, ylab = 'x (mm)', 'y (mm)', 'mm'
    else:
        xlab, ylab_xy, ylab = 'x', 'y', 'mm'

    t, dist, valid, events, pad = _events_from_dist(x1,y1,x2,y2,fps,dist_thresh)
    valid_seg = valid[:-1] & valid[1:]

    xmin, xmax = np.nanmin([x1,x2]), np.nanmax([x1,x2])
    ymin, ymax = np.nanmin([y1,y2]), np.nanmax([y1,y2])

    # tight layout: close gaps
    fig = plt.figure(figsize=(12, 6.4), constrained_layout=True)
    gs = fig.add_gridspec(2,2, width_ratios=[1.2,1.0], height_ratios=[1,1])
    fig.subplots_adjust(hspace=0.12, wspace=0.12)

    ax_d_plain = fig.add_subplot(gs[0,0])
    ax_xy_plain= fig.add_subplot(gs[0,1])
    ax_d_evt   = fig.add_subplot(gs[1,0], sharex=ax_d_plain)  # share x with TL
    ax_xy_evt  = fig.add_subplot(gs[1,1])

    def _min_style(ax, left=False, keep_xticks=False):
        for s in ('top','right'): ax.spines[s].set_visible(False)
        if not keep_xticks:
            ax.tick_params(axis='x', labelbottom=False, length=0)
        ax.tick_params(axis='y', length=(3 if left else 0), labelleft=left)

    # ---- TL: Inter-animal COM distance (with time ticks) ----
    ax_d_plain.plot(t, dist, lw=1.0, color='C0')
    ax_d_plain.set_title('Inter-animal COM distance', fontsize=11, pad=4)
    ax_d_plain.set_ylabel(ylab)
    ax_d_plain.set_xlabel('time (s)')  # show x ticks on top distance too
    _min_style(ax_d_plain, left=True, keep_xticks=True)

    # ---- TR: COM trajectories (plain) ----
    if valid_seg.any():
        ax_xy_plain.add_collection(LineCollection(_seg_lines(x1,y1,valid_seg),
                               colors=animal_colors[0], linewidths=1.0, alpha=0.45))
        ax_xy_plain.add_collection(LineCollection(_seg_lines(x2,y2,valid_seg),
                               colors=animal_colors[1], linewidths=1.0, alpha=0.45))
    ax_xy_plain.set_xlim(xmin, xmax); ax_xy_plain.set_ylim(ymin, ymax)
    ax_xy_plain.set_aspect('equal','box')
    ax_xy_plain.set_title('COM trajectories', fontsize=11, pad=4)
    ax_xy_plain.set_xlabel(xlab); ax_xy_plain.set_ylabel(ylab_xy)
    _min_style(ax_xy_plain, left=False, keep_xticks=False)

    # ---- BL: Inter-animal distance (xy) with events ----
    ax_d_evt.plot(t, dist, lw=1.2, color='0.15')
    ax_d_evt.axhline(dist_thresh, ls='--', lw=0.9, color='k', alpha=0.7)
    cmap = cm.get_cmap('tab10', max(1,len(events)))
    for k, ev in enumerate(events, start=1):
        ax_d_evt.axvspan(ev['t0'], ev['t1'], color=cmap((k-1) % 10), alpha=0.18)
    ax_d_evt.set_title('Inter-animal distance (xy)', fontsize=11, pad=4)
    ax_d_evt.set_xlabel('time (s)'); ax_d_evt.set_ylabel(ylab)
    _min_style(ax_d_evt, left=True, keep_xticks=True)

    # ---- BR: COM trajectories (events) ----
    if valid_seg.any():
        ax_xy_evt.add_collection(LineCollection(_seg_lines(x1,y1,valid_seg),
                             colors=animal_colors[0], linewidths=1.0, alpha=0.15))
        ax_xy_evt.add_collection(LineCollection(_seg_lines(x2,y2,valid_seg),
                             colors=animal_colors[1], linewidths=1.0, alpha=0.15))
        for k, ev in enumerate(events, start=1):
            s = max(0, ev['start']-pad); e = min(T-1, ev['end']+pad)
            win = np.zeros_like(valid_seg); win[s:e]=True
            mask = valid_seg & win
            col = cmap((k-1) % 10)
            if mask.any():
                ax_xy_evt.add_collection(LineCollection(_seg_lines(x1,y1,mask),
                                     colors=col, linewidths=2.2, alpha=0.95))
                ax_xy_evt.add_collection(LineCollection(_seg_lines(x2,y2,mask),
                                     colors=col, linewidths=2.2, alpha=0.95))
            if annotate_meetings:
                idx = ev['rep']
                if 0<=idx<T and valid[idx]:
                    mx = 0.5*(x1[idx]+x2[idx]); my = 0.5*(y1[idx]+y2[idx])
                    ax_xy_evt.text(mx, my, str(k), fontsize=9, ha='center', va='center',
                                   bbox=dict(boxstyle="circle,pad=0.15", fc='white',
                                             ec=col, lw=0.9, alpha=0.95))
    ax_xy_evt.set_xlim(xmin, xmax); ax_xy_evt.set_ylim(ymin, ymax)
    ax_xy_evt.set_aspect('equal','box')
    ax_xy_evt.set_title('COM trajectories (events)', fontsize=11, pad=4)
    ax_xy_evt.set_xlabel(xlab); ax_xy_evt.set_ylabel(ylab_xy)
    _min_style(ax_xy_evt, left=False, keep_xticks=False)

    if save:
        fig.savefig(os.path.join(base_path, out_png), dpi=300, bbox_inches='tight')
    if not show:
        plt.close(fig)

    return dict(time=t, dist=dist, events=events, xy_limits=(xmin,xmax,ymin,ymax))


In [ ]:
p = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30"
plot_social_quad_poster(p, fps=30.0, dist_thresh=120.0, annotate_meetings=False, save=False, show=True)


In [ ]:
import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Patch

def plot_com_occupancy_heatmaps(
    base_path,
    pred_folder='DANNCE/predict00',
    com_dir='yolo_tracking/predict_train_lone_social_retrain_0917',
    vis_dir='COM/predict00/vis',
    pair=(0, 1),
    bins=100,                 # int or (nx, ny)
    px_to_mm=None,            # scalar to label axes in mm
    normalize=True,           # prob vs counts (for single-animal maps)
    smooth_sigma=1.0,         # Gaussian blur in bin units (0 disables)
    vmax_pct=99.5,            # single-animal color scaling (nonzero percentile)
    gamma=0.6,                # single-animal gamma for low-density lift
    # overlap intensity (built from RAW, not smoothed)
    overlap_low_pct=70,       # drop low tail
    overlap_high_pct=99.5,    # cap high tail
    overlap_gamma=0.8,        # nonlinearity for overlap intensity
    overlap_imax=0.85,        # max intensity per animal in overlap (0..1)
    animal_colors=('C0','C1'),
    titles=('Occupancy — Animal A','Occupancy — Animal B','Occupancy — Overlap'),
    filenames=('occ_A.png','occ_B.png','occ_overlap.png'),
    show=True,
    save=False,
    # --- new poster options ---
    layout='separate',        # 'separate' (old behavior) or 'triptych' (side-by-side)
    poster=True,              # minimal markings if True (used only when layout='triptych')
    show_ticks=False,         # hide tick marks/labels in poster mode
    show_colorbars=False,     # hide colorbars in poster mode
    panel_labels=('A','B','A∩B'),
    triptych_filename='occ_triptych.png',
):
    # ---------- load COM → (T,2,N) ----------
    primary = os.path.join(base_path, pred_folder, 'com3d_used.mat')
    fallback = os.path.join(base_path, com_dir, 'com3d0.mat')
    com_file = primary if os.path.isfile(primary) else fallback
    raw = sio.loadmat(com_file).get('com')
    if raw is None:
        raise KeyError(f"'com' not found in {com_file}")

    if raw.ndim == 3 and raw.shape[1] == 3:
        com_xy = raw[:, :2, :]
    elif raw.ndim == 3 and raw.shape[1] == 2:
        com_xy = raw
    elif raw.ndim == 2 and raw.shape[1] % 3 == 0:
        N = raw.shape[1] // 3
        com_xy = raw.reshape(-1, 3, N)[:, :2, :]
    elif raw.ndim == 2 and raw.shape[1] % 2 == 0:
        N = raw.shape[1] // 2
        com_xy = raw.reshape(-1, 2, N)
    else:
        raise ValueError(f"Unexpected COM shape {raw.shape}")

    T, _, N = com_xy.shape
    if max(pair) >= N:
        raise ValueError(f"Pair index {pair} exceeds #animals {N}")

    x1, y1 = com_xy[:, 0, pair[0]], com_xy[:, 1, pair[0]]
    x2, y2 = com_xy[:, 0, pair[1]], com_xy[:, 1, pair[1]]
    v = np.isfinite(x1) & np.isfinite(y1) & np.isfinite(x2) & np.isfinite(y2)
    x1, y1, x2, y2 = x1[v], y1[v], x2[v], y2[v]

    ulabel = ''
    if px_to_mm is not None:
        s = float(px_to_mm)
        x1, y1, x2, y2 = x1*s, y1*s, x2*s, y2*s
        ulabel = ' (mm)'

    if x1.size == 0 or x2.size == 0:
        raise ValueError("No valid positions for one or both animals.")

    # ---------- shared bin edges ----------
    xmin = float(np.nanmin([x1.min(), x2.min()]))
    xmax = float(np.nanmax([x1.max(), x2.max()]))
    ymin = float(np.nanmin([y1.min(), y2.min()]))
    ymax = float(np.nanmax([y1.max(), y2.max()]))

    if isinstance(bins, int):
        nx = ny = int(bins)
    else:
        nx, ny = int(bins[0]), int(bins[1])

    xedges = np.linspace(xmin, xmax, nx+1)
    yedges = np.linspace(ymin, ymax, ny+1)
    extent = (xedges[0], xedges[-1], yedges[0], yedges[-1])

    # ---------- histograms (keep RAW for overlap) ----------
    H1_raw, _, _ = np.histogram2d(x1, y1, bins=[xedges, yedges])
    H2_raw, _, _ = np.histogram2d(x2, y2, bins=[xedges, yedges])

    H1, H2 = H1_raw.copy(), H2_raw.copy()
    if smooth_sigma and smooth_sigma > 0:
        H1 = gaussian_filter(H1, sigma=smooth_sigma, mode='nearest')
        H2 = gaussian_filter(H2, sigma=smooth_sigma, mode='nearest')

    # ---------- single-animal scaling ----------
    if normalize:
        s1, s2 = H1.sum(), H2.sum()
        if s1 > 0: H1 /= s1
        if s2 > 0: H2 /= s2
        cbar_label = 'occupancy (prob.)'
    else:
        cbar_label = 'occupancy (frames)'

    def _scale_for_display(H):
        nz = H[H > 0]
        vmax = np.percentile(nz, vmax_pct) if nz.size else 1.0
        return mpl.colors.Normalize(vmin=0, vmax=vmax)

    norm1 = _scale_for_display(H1)
    norm2 = _scale_for_display(H2)

    # ---------- overlap: simple transparent overlay (clean + robust) ----------
    H1m = np.ma.masked_less_equal(H1, 0)
    H2m = np.ma.masked_less_equal(H2, 0)

    # ---------- draw ----------
    if layout == 'separate':
        # old behavior: three figures
        figA, axA = plt.subplots(figsize=(5.2, 5.2))
        imA = axA.imshow(H1.T, origin='lower', extent=extent, cmap='Blues', norm=norm1, interpolation='nearest')
        axA.set_aspect('equal'); axA.set_title(titles[0] + f' — {os.path.basename(base_path)}')
        axA.set_xlabel('x'+ulabel); axA.set_ylabel('y'+ulabel)
        cbarA = figA.colorbar(imA, ax=axA, fraction=0.046, pad=0.04); cbarA.set_label(cbar_label)

        figB, axB = plt.subplots(figsize=(5.2, 5.2))
        imB = axB.imshow(H2.T, origin='lower', extent=extent, cmap='Oranges', norm=norm2, interpolation='nearest')
        axB.set_aspect('equal'); axB.set_title(titles[1] + f' — {os.path.basename(base_path)}')
        axB.set_xlabel('x'+ulabel); axB.set_ylabel('y'+ulabel)
        cbarB = figB.colorbar(imB, ax=axB, fraction=0.046, pad=0.04); cbarB.set_label(cbar_label)

        figC, axC = plt.subplots(figsize=(6.0, 6.0))
        axC.set_aspect('equal', 'box'); axC.set_facecolor('white')
        axC.set_title(titles[2] + f' — {os.path.basename(base_path)}')
        axC.set_xlabel('x' + ulabel); axC.set_ylabel('y' + ulabel)
        axC.imshow(H1m.T, origin='lower', extent=extent, cmap='Blues',   norm=norm1, interpolation='nearest', alpha=0.55)
        axC.imshow(H2m.T, origin='lower', extent=extent, cmap='Oranges', norm=norm2, interpolation='nearest', alpha=0.55)

        if save:
            out_dir = os.path.join(base_path, vis_dir); os.makedirs(out_dir, exist_ok=True)
            figA.tight_layout(); figB.tight_layout(); figC.tight_layout()
            figA.savefig(os.path.join(out_dir, filenames[0]), dpi=300)
            figB.savefig(os.path.join(out_dir, filenames[1]), dpi=300)
            figC.savefig(os.path.join(out_dir, filenames[2]), dpi=300)

        if not show:
            plt.close(figA); plt.close(figB); plt.close(figC)

    elif layout == 'triptych':
        # poster-ready: a single row, minimal markings
        # small, consistent fonts; clean background
        if poster:
            mpl.rcParams.update({
                "axes.edgecolor": "0.15",
                "axes.linewidth": 0.6,
                "font.size": 9,
                "axes.titlesize": 10,
                "xtick.major.size": 0, "ytick.major.size": 0,
            })

        fig, axes = plt.subplots(1, 3, figsize=(14.4, 4.8), constrained_layout=True)
        axA, axB, axC = axes

        # A
        imA = axA.imshow(H1.T, origin='lower', extent=extent, cmap='Blues', norm=norm1, interpolation='nearest')
        axA.set_aspect('equal'); 
        # B
        imB = axB.imshow(H2.T, origin='lower', extent=extent, cmap='Oranges', norm=norm2, interpolation='nearest')
        axB.set_aspect('equal')
        # C (overlap)
        axC.imshow(H1m.T, origin='lower', extent=extent, cmap='Blues',   norm=norm1, interpolation='nearest', alpha=0.55)
        axC.imshow(H2m.T, origin='lower', extent=extent, cmap='Oranges', norm=norm2, interpolation='nearest', alpha=0.55)
        axC.set_aspect('equal')

        # unified limits and minimal axes
        for ax in axes:
            ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
            if poster:
                ax.set_title('')
                if not show_ticks:
                    ax.set_xticks([]); ax.set_yticks([])
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.spines['left'].set_visible(False)
                ax.spines['bottom'].set_visible(False)
            else:
                ax.set_xlabel('x'+ulabel); ax.set_ylabel('y'+ulabel)

        # tiny in-panel labels (non-intrusive)
        if panel_labels:
            for lab, ax in zip(panel_labels, axes):
                ax.text(0.02, 0.98, lab, transform=ax.transAxes, ha='left', va='top',
                        fontsize=10, fontweight='bold', color='0.15')

        # optional colorbars (off by default)
        if show_colorbars:
            # narrow colorbars beneath A and B only
            from mpl_toolkits.axes_grid1 import make_axes_locatable
            for ax, im in [(axA, imA), (axB, imB)]:
                divider = make_axes_locatable(ax)
                cax = divider.append_axes("bottom", size="3%", pad=0.5)
                cb = fig.colorbar(im, cax=cax, orientation='horizontal')
                cb.set_label(cbar_label)

        if save:
            out_dir = os.path.join(base_path, vis_dir); os.makedirs(out_dir, exist_ok=True)
            fig.savefig(os.path.join(out_dir, triptych_filename), dpi=300, bbox_inches='tight')

        if not show:
            plt.close(fig)

    else:
        raise ValueError("layout must be 'separate' or 'triptych'")

    return {
        'H1': H1, 'H2': H2,
        'H1_raw': H1_raw, 'H2_raw': H2_raw,
        'xedges': xedges, 'yedges': yedges, 'extent': extent,
        'norm1': norm1, 'norm2': norm2
    }


In [ ]:
occ = plot_com_occupancy_heatmaps(
    base_path=p,
    bins=80,
    normalize=True,
    smooth_sigma=1.2,
    vmax_pct=99.5, gamma=0.65,
    overlap_low_pct=75, overlap_high_pct=99.2,
    overlap_gamma=0.9, overlap_imax=0.8,
    animal_colors=('C0','C1'),
    layout='triptych',          # <- side-by-side
    poster=True,                # <- minimal markings
    show_ticks=False,
    show_colorbars=False,
    save=False, show=True
)
